In [10]:
#r "ServerThreadTests/bin/Debug/net10.0/task17.dll"
#r "nuget:ScottPlot,5.0.54"

using System;
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Threading;
using ScottPlot;
using task17;

public class LightCommand : ILongCommand
{
    private int _remaining;
    private readonly int _iterations;
    private double _result;
    
    public bool IsCompleted => _remaining <= 0;
    public string Name { get; }
    public int ExecutedCount { get; private set; }
    
    public LightCommand(int requiredCalls, string name = "", int iterations = 50000)
    {
        _remaining = requiredCalls;
        _iterations = iterations;
        Name = name;
        ExecutedCount = 0;
    }
    
    public void Execute()
    {
        if (IsCompleted) return;
        
        double x = 0;
        for (int i = 0; i < _iterations; i++)
            x += Math.Sin(i) * Math.Cos(i);
        
        _result = x;
        _remaining--;
        ExecutedCount++;
    }
}

Console.WriteLine("ЗАМЕРЫ ПРОИЗВОДИТЕЛЬНОСТИ");
Console.WriteLine(new string('=', 50));
Console.WriteLine();

var counts = new int[] { 1, 2, 4, 8, 16 };
var callsPerCommand = 10;
var singleTimes = new double[counts.Length];
var rrTimes = new double[counts.Length];

for (int i = 0; i < counts.Length; i++)
{
    int n = counts[i];

    double singleTotal = 0;
    for (int run = 0; run < 3; run++)
    {
        var commands = new LightCommand[n];
        for (int j = 0; j < n; j++)
            commands[j] = new LightCommand(callsPerCommand);

        var sw = Stopwatch.StartNew();
        foreach (var cmd in commands)
        {
            while (!cmd.IsCompleted)
                cmd.Execute();
        }
        sw.Stop();
        singleTotal += sw.Elapsed.TotalMilliseconds;
    }
    singleTimes[i] = singleTotal / 3;

    double rrTotal = 0;
    for (int run = 0; run < 3; run++)
    {
        var scheduler = new RoundRobinScheduler();
        var server = new ServerThread(scheduler);
        var commands = new LightCommand[n];
        
        for (int j = 0; j < n; j++)
        {
            commands[j] = new LightCommand(callsPerCommand);
            server.Add(commands[j]);
        }

        server.Start();
        var sw = Stopwatch.StartNew();

        while (commands.Any(c => !c.IsCompleted))
            Thread.Sleep(1);

        sw.Stop();
        server.SoftStop();
        server.Join();
        rrTotal += sw.Elapsed.TotalMilliseconds;
    }
    rrTimes[i] = rrTotal / 3;

    Console.WriteLine($"Команд: {n,2} | Последовательно: {singleTimes[i],8:F1} мс | Round Robin: {rrTimes[i],8:F1} мс");
}

var plot = new Plot();
plot.Title("Производительность: последовательно vs Round Robin");
plot.XLabel("Количество команд");
plot.YLabel("Время выполнения (мс)");

var xs = counts.Select(x => (double)x).ToArray();

var scatter1 = plot.Add.Scatter(xs, singleTimes);
scatter1.LegendText = "Последовательно";
scatter1.LineWidth = 2;
scatter1.MarkerSize = 8;

var scatter2 = plot.Add.Scatter(xs, rrTimes);
scatter2.LegendText = "Round Robin (ServerThread)";
scatter2.LineWidth = 2;
scatter2.MarkerSize = 8;

plot.ShowLegend();

var plotPath = "plot.png";
plot.SavePng(plotPath, 800, 600);
display(HTML($"<img src='{plotPath}?t={DateTime.Now.Ticks}' width='700'/>"));

var reportPath = "report.txt";
var sb = new System.Text.StringBuilder();

sb.AppendLine($"Дата: {DateTime.Now}");
sb.AppendLine();
sb.AppendLine("1. Реализованы RoundRobinScheduler и ServerThread");
sb.AppendLine("2. Замеры (LightCommand, 50K итераций x 10 вызовов):");
for (int i = 0; i < counts.Length; i++)
    sb.AppendLine($"   Команд: {counts[i],2} | Последовательно: {singleTimes[i],7:F1} мс | Round Robin: {rrTimes[i],7:F1} мс");
sb.AppendLine();
sb.AppendLine("3. Round Robin выполняет длительные операции");
sb.AppendLine("   без монопольной блокировки потока.");

File.WriteAllText(reportPath, sb.ToString());
Console.WriteLine($"Отчёт сохранён: {reportPath}");

Installed Packages ScottPlot, 5.0.54

ЗАМЕРЫ ПРОИЗВОДИТЕЛЬНОСТИ

Команд:  1 | Последовательно:     20.4 мс | Round Robin:     16.2 мс
Команд:  2 | Последовательно:     42.0 мс | Round Robin:     29.7 мс
Команд:  4 | Последовательно:     51.6 мс | Round Robin:     52.3 мс
Команд:  8 | Последовательно:     93.3 мс | Round Robin:    100.2 мс
Команд: 16 | Последовательно:    178.1 мс | Round Robin:    188.1 мс


Отчёт сохранён: report.txt
